# Decoder Evaluation: Threshold Sweep

Evaluates the decoder under three embedding conditions and sweeps binarization thresholds
to find the best precision/recall trade-off.

**Conditions tested:**
1. **Clean** — raw encoder embeddings (upper bound)
2. **PCA roundtrip** — compress+decompress via PCA (matches decoder training distribution)
3. **PCA roundtrip + noise** — adds `pca_aug_noise_std` Gaussian noise (most realistic for generation)


In [ ]:
#| default_exp eval_dec

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from omegaconf import DictConfig
from tqdm.auto import tqdm

from midi_rae.core import HierarchicalPatchState, EncoderOutput
from midi_rae.swin import SwinDecoder
from midi_rae.train_dec import setup_models, load_pca_models, pca_roundtrip_enc_out
from midi_rae.data import PreEncodedChunkDataset, collate_preencode, emb_levels_to_enc_out
from midi_rae.utils import load_checkpoint, cjprint

In [ ]:
#| export
@torch.no_grad()
def collect_probs(decoder, val_dl, device, pca_models=None, n_aug_levels=0, noise_std=0.0,
                  n_batches=None):
    """Run decoder on val set and collect sigmoid probabilities + ground truth.

    Returns:
        probs: (N,) tensor of per-pixel sigmoid probabilities (flattened)
        reals: (N,) tensor of ground truth binary values (flattened)
    """
    decoder.eval()
    all_probs, all_reals = [], []
    for i, batch in enumerate(tqdm(val_dl, desc='collecting probs')):
        if n_batches is not None and i >= n_batches:
            break
        img_real = batch['img'].to(device)  # (B, 1, H, W)
        enc_out = emb_levels_to_enc_out(batch, device)
        if pca_models and n_aug_levels > 0:
            enc_out = pca_roundtrip_enc_out(enc_out, pca_models, n_aug_levels, device, noise_std=noise_std)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = decoder(enc_out)           # (B, 1, H, W), pre-sigmoid
        probs = torch.sigmoid(logits.float())   # (B, 1, H, W)
        all_probs.append(probs.cpu().flatten())
        all_reals.append(img_real.cpu().flatten())
    return torch.cat(all_probs), torch.cat(all_reals)

In [ ]:
#| export
def threshold_sweep(probs, reals, thresholds=None, eps=1e-8):
    """Compute precision, recall, F1 at each threshold.

    Args:
        probs: (N,) sigmoid probabilities
        reals: (N,) binary ground truth
        thresholds: list/array of thresholds to sweep (default: 0.05 to 0.95)

    Returns dict with keys 'thresholds', 'precision', 'recall', 'f1'.
    """
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 37)
    reals = reals.bool()
    results = {'thresholds': thresholds, 'precision': [], 'recall': [], 'f1': []}
    for t in thresholds:
        pred = probs >= t
        tp = (pred & reals).sum().item()
        fp = (pred & ~reals).sum().item()
        fn = (~pred & reals).sum().item()
        prec = tp / (tp + fp + eps)
        rec  = tp / (tp + fn + eps)
        f1   = 2 * prec * rec / (prec + rec + eps)
        results['precision'].append(prec)
        results['recall'].append(rec)
        results['f1'].append(f1)
    for k in ('precision', 'recall', 'f1'):
        results[k] = np.array(results[k])
    return results

In [ ]:
#| export
def plot_threshold_curves(results_by_condition, output_path=None):
    """Plot precision, recall, F1 vs threshold for each condition."""
    colors = {'clean': 'steelblue', 'pca': 'darkorange', 'pca+noise': 'forestgreen'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: precision & recall vs threshold
    ax = axes[0]
    for cond, res in results_by_condition.items():
        c = colors.get(cond, 'gray')
        t = res['thresholds']
        ax.plot(t, res['precision'], c=c, linestyle='-',  label=f'{cond} precision')
        ax.plot(t, res['recall'],    c=c, linestyle='--', label=f'{cond} recall')
        ax.plot(t, res['f1'],        c=c, linestyle=':',  label=f'{cond} F1')
        best_idx = np.argmax(res['f1'])
        ax.axvline(t[best_idx], color=c, alpha=0.3)
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Precision / Recall / F1 vs Threshold\n(solid=prec, dashed=rec, dotted=F1)')
    ax.legend(fontsize=7)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

    # Right: precision-recall curve
    ax = axes[1]
    for cond, res in results_by_condition.items():
        c = colors.get(cond, 'gray')
        ax.plot(res['recall'], res['precision'], c=c, label=cond)
        best_idx = np.argmax(res['f1'])
        ax.scatter(res['recall'][best_idx], res['precision'][best_idx],
                   c=c, zorder=5, s=60,
                   label=f"{cond} best F1={res['f1'][best_idx]:.3f} @t={res['thresholds'][best_idx]:.2f}")
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve')
    ax.legend(fontsize=7)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f'Saved to {output_path}')
    plt.show()
    return fig

In [ ]:
#| export
def eval_decoder(cfg: DictConfig, n_batches=None, output_dir=None):
    """Evaluate decoder threshold sweep under clean / PCA / PCA+noise conditions.

    Loads the decoder from cfg.training.dec_init_ckpt (or cfg.generate.decoder_ckpt),
    runs inference on the val split of the pre-encoded dataset, sweeps thresholds,
    and saves a precision-recall plot.

    Args:
        cfg: Hydra DictConfig
        n_batches: limit eval to this many val batches (None = full val set)
        output_dir: where to save the plot (default: outputs/eval_dec/)
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cjprint(f'Evaluating decoder on {device}', color='cyan')

    # --- decoder ---
    dec_ckpt = cfg.training.get('dec_init_ckpt', None) or cfg.generate.get('decoder_ckpt', None)
    assert dec_ckpt, 'Set cfg.training.dec_init_ckpt or cfg.generate.decoder_ckpt'
    dec_ckpt = os.path.expandvars(os.path.expanduser(str(dec_ckpt)))
    m = cfg.model
    decoder = SwinDecoder(
        img_height=cfg.data.image_size, img_width=cfg.data.image_size,
        patch_h=m.patch_h, patch_w=m.patch_w, out_channels=cfg.data.in_channels,
        embed_dim=m.embed_dim, depths=list(m.dec_depths), num_heads=list(m.dec_num_heads),
        window_size=m.window_size, mlp_ratio=m.mlp_ratio, drop_path_rate=0.0,
    )
    decoder = load_checkpoint(decoder, dec_ckpt).to(device).eval()
    cjprint(f'Loaded decoder from {dec_ckpt}', color='green')

    # --- val dataloader ---
    encoded_dir = os.path.expandvars(os.path.expanduser(str(cfg.preencode.output_dir)))
    val_ds = PreEncodedChunkDataset(encoded_dir, split='val')
    val_dl = DataLoader(val_ds, batch_size=cfg.training.dec_batch_size,
                        shuffle=False, num_workers=2, persistent_workers=True,
                        collate_fn=collate_preencode)
    cjprint(f'Val set: {len(val_ds)} samples', color='cyan')

    # --- PCA models ---
    pca_dir = os.path.expandvars(os.path.expanduser(str(cfg.training.pca_dir)))
    n_aug_levels = cfg.training.get('pca_aug_levels', 6)
    noise_std    = cfg.training.get('pca_aug_noise_std', 0.05)
    fine_levels       = list(cfg.training.get('pca_fine_levels', []))
    fine_n_components = list(cfg.training.get('pca_fine_n_components', []))
    pca_models = load_pca_models(pca_dir, n_aug_levels,
                                 fine_levels=fine_levels, fine_n_components=fine_n_components)
    cjprint(f'Loaded {len(pca_models)} PCA models (levels {sorted(pca_models.keys())})', color='cyan')

    # --- collect probs for each condition ---
    conditions = {
        'clean':     dict(pca_models=None,       n_aug_levels=0,           noise_std=0.0),
        'pca':       dict(pca_models=pca_models,  n_aug_levels=n_aug_levels, noise_std=0.0),
        'pca+noise': dict(pca_models=pca_models,  n_aug_levels=n_aug_levels, noise_std=noise_std),
    }
    results_by_condition = {}
    for cond, kwargs in conditions.items():
        cjprint(f'\nRunning condition: {cond}', color='yellow')
        probs, reals = collect_probs(decoder, val_dl, device, n_batches=n_batches, **kwargs)
        res = threshold_sweep(probs, reals)
        results_by_condition[cond] = res
        best_idx = np.argmax(res['f1'])
        print(f'  Best F1={res["f1"][best_idx]:.4f}  '
              f'prec={res["precision"][best_idx]:.4f}  '
              f'rec={res["recall"][best_idx]:.4f}  '
              f'@ threshold={res["thresholds"][best_idx]:.3f}')

    # --- plot ---
    out_dir = Path(os.path.expandvars(os.path.expanduser(output_dir or 'outputs/eval_dec')))
    out_dir.mkdir(parents=True, exist_ok=True)
    plot_path = out_dir / 'threshold_sweep.png'
    plot_threshold_curves(results_by_condition, output_path=str(plot_path))

    return results_by_condition

In [ ]:
#| export
#| eval: false
import hydra

@hydra.main(version_base=None, config_path='../configs', config_name='config_swin')
def eval_dec_main(cfg: DictConfig):
    n_batches = cfg.get('eval_n_batches', None)
    output_dir = cfg.get('eval_output_dir', 'outputs/eval_dec')
    eval_decoder(cfg, n_batches=n_batches, output_dir=output_dir)

if __name__ == '__main__':
    eval_dec_main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()